In [1]:
%pip install -q sentence-transformers datasets accelerate

In [2]:
import pandas as pd
import os

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses
)

from torch.utils.data import DataLoader

/tmp/ipykernel_3552/153941063.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
DATA_PATH = r"/content/drive/MyDrive/immverse.ai/triplets.csv"

MODEL_NAME = "intfloat/multilingual-e5-small"

SAVE_MODEL_PATH = r"/content/drive/MyDrive/immverse.ai/models_10_epochs"

os.makedirs(SAVE_MODEL_PATH, exist_ok=True)

In [5]:
triplets = pd.read_csv(DATA_PATH)

triplets.head()

,anchor,positive,negative
0,What does this verse say?,Verse ID: 137\n\nSanskrit:\nप्रकृतेः क्रियमाणा...,Verse ID: 67\n\nSanskrit:\nअव्यक्तादीनि भूतानि...
1,Explain verse 137,Verse ID: 137\n\nSanskrit:\nप्रकृतेः क्रियमाणा...,Verse ID: 182\n\nSanskrit:\nअपाने जुह्वति प्रा...
2,Explain verse 326,Verse ID: 326\n\nSanskrit:\nमया ततमिदं सर्वं ज...,Verse ID: 25\n\nSanskrit:\nअर्जुन उवाचदृष्ट्वे...
3,What does this verse say?,Verse ID: 547\n\nSanskrit:\nअनेकचित्तविभ्रान्त...,Verse ID: 174\n\nSanskrit:\nनिराशीर्यतचित्तात्...
4,एतान्यपि तु कर्माणि सङ्गं त्यक्त्वा फलानि च ।क...,Verse ID: 587\n\nSanskrit:\nएतान्यपि तु कर्माण...,Verse ID: 121\n\nSanskrit:\nदेवान्भावयतातेन ते...


In [6]:
print("Total Triplets :", len(triplets))
triplets.sample(5)

Total Triplets : 3942


,anchor,positive,negative
3258,What does this verse say?,Verse ID: 588\n\nSanskrit:\nनियतस्य तु सन्न्या...,Verse ID: 100\n\nSanskrit:\nध्यायतो विषयान्पुं...
1916,ये चैव सात्त्विका भावा राजसास्तामसाश्च ये ।मत्...,Verse ID: 276\n\nSanskrit:\nये चैव सात्त्विका ...,Verse ID: 125\n\nSanskrit:\nकर्म ब्रह्मोद्भ‍वं...
1626,What is explained in verse 113?,Verse ID: 113\n\nSanskrit:\nश्रीभगवानुवाचलोकेऽ...,Verse ID: 280\n\nSanskrit:\nचतुर्विधा भजन्ते म...
3880,“Persons who are engaged in the worship of dem...,Verse ID: 345\n\nSanskrit:\nयेऽप्यन्यदेवताभक्त...,Verse ID: 621\n\nSanskrit:\nन तदस्ति पृथिव्यां...
1149,Meaning of verse 579,Verse ID: 579\n\nSanskrit:\nतदित्यनभिसन्धाय फल...,Verse ID: 262\n\nSanskrit:\nप्रयत्‍नाद्यतमानस्...


In [7]:
model = SentenceTransformer(MODEL_NAME)

print("Loaded Model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded Model


In [8]:
train_examples = []

for _, row in triplets.iterrows():

    train_examples.append(

        InputExample(
            texts=[
                row["anchor"],
                row["positive"],
                row["negative"]
            ]
        )

    )

print(len(train_examples))

3942


In [9]:
train_dataloader = DataLoader(

    train_examples,

    shuffle=True,

    batch_size=16

)

In [10]:
train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
NUM_EPOCHS = 2

WARMUP = int(len(train_dataloader) * NUM_EPOCHS * 0.1)

model.fit(

    train_objectives=[
        (train_dataloader, train_loss)
    ],

    epochs=NUM_EPOCHS,

    warmup_steps=WARMUP,

    output_path=SAVE_MODEL_PATH,

    show_progress_bar=True

)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
model.save(SAVE_MODEL_PATH)

print("Model Saved Successfully")

In [ ]:
trained_model = SentenceTransformer(SAVE_MODEL_PATH)

print("Reload Successful")

Test Embeddings

In [ ]:
query = "What is karma?"

embedding = trained_model.encode(query)

print(embedding.shape)

In [ ]:
doc1 = "कर्मण्येवाधिकारस्ते"

doc2 = "You have a right to perform your duties."

doc3 = "Quantum Mechanics"

embeddings = trained_model.encode(
    [query, doc1, doc2, doc3],
    normalize_embeddings=True
)

In [ ]:
from sentence_transformers.util import cos_sim

query_embedding = embeddings[0]

print("Query ↔ Sanskrit")

print(
    cos_sim(query_embedding, embeddings[1]).item()
)

print()

print("Query ↔ English")

print(
    cos_sim(query_embedding, embeddings[2]).item()
)

print()

print("Query ↔ Quantum")

print(
    cos_sim(query_embedding, embeddings[3]).item()
)